# Table 2 (Instruction-Consistent Build)

This notebook rebuilds Table 2 using the three-step construction in `src/data instruction.txt`:
1. Build issuer totals (from the existing `table1.ipynb` construction pipeline).
2. Build foreign holdings by investor from IMF PIP bilateral data.
3. Construct domestic holdings as issuer totals minus foreign holdings, then assign domestic holdings to the home investor.

Important implementation notes:
- No hard-coded target values are used.
- Confidential/small positive holdings (`0 < value_usd < 500000`) are set to missing before aggregation.
- A separate pseudo-investor row `Reserves` is added from reserve-sector bilateral positions.
- Consistency checks are printed so discrepancies are explicit and auditable.

In [1]:
from pathlib import Path
from typing import Optional
import json
import pandas as pd

In [2]:
TARGET_YEAR = 2020
SMALL_HOLDING_USD = 500_000.0

# Appendix C offshore financial centers (investor side to avoid double counting).
OFC_ISO3 = {"BMU", "CYM", "GGY", "IRL", "IMN", "JEY", "LUX", "AN", "ANT"}

project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent

data_dir = project_root / "data"
src_dir = project_root / "src"
table1_nb_path = src_dir / "table1.ipynb"
bilat_path = data_dir / "pip_bilateral_positions.parquet"
local_alloc_path = data_dir / "pip_local_foreign_allocated.parquet"
reserve_path = data_dir / "pip_bilateral_positions_reserve.parquet"
reserve_agg_path = data_dir / "pip_bilateral_positions_reserve_aggregate.parquet"

for p in [table1_nb_path, bilat_path, local_alloc_path, reserve_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

ASSET_CODE_TO_LABEL = {
    "ST_DEBT": "Short-term debt",
    "LT_DEBT": "Long-term debt",
    "EQUITY": "Equity",
}

ASSET_LABEL_TO_MN_COL = {
    "ST_DEBT": "st_mn",
    "LT_DEBT": "lt_mn",
    "EQUITY": "eq_mn",
}

ASSET_LABEL_TO_FOREIGN_MN_COL = {
    "ST_DEBT": "foreign_st_mn",
    "LT_DEBT": "foreign_lt_mn",
    "EQUITY": "foreign_eq_mn",
}

ISO3_TO_NAME = {
    "CAN": "Canada", "USA": "United States", "AUT": "Austria", "BEL": "Belgium", "DNK": "Denmark",
    "FIN": "Finland", "FRA": "France", "DEU": "Germany", "ISR": "Israel", "ITA": "Italy",
    "NLD": "Netherlands", "NOR": "Norway", "PRT": "Portugal", "ESP": "Spain", "SWE": "Sweden",
    "CHE": "Switzerland", "GBR": "United Kingdom", "AUS": "Australia", "HKG": "Hong Kong",
    "JPN": "Japan", "NZL": "New Zealand", "SGP": "Singapore", "BRA": "Brazil",
    "CHN": "China", "COL": "Colombia", "CZE": "Czech Republic", "GRC": "Greece",
    "HUN": "Hungary", "IND": "India", "MYS": "Malaysia", "MEX": "Mexico",
    "PHL": "Philippines", "POL": "Poland", "RUS": "Russia", "ZAF": "South Africa",
    "KOR": "South Korea", "THA": "Thailand",
}

In [3]:
def _load_panel_from_table1_notebook(nb_path: Path, target_year: int) -> pd.DataFrame:
    """Execute setup cells from table1 notebook and return the year-specific panel."""
    nb = json.loads(nb_path.read_text(encoding="utf-8-sig"))
    code_cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    if len(code_cells) < 2:
        raise RuntimeError("table1.ipynb has fewer than 2 code cells; cannot reuse totals pipeline.")

    ns = {}
    for c in code_cells[:2]:
        src = "\n".join(c.get("source", []))
        if src.strip():
            exec(compile(src, str(nb_path), "exec"), ns, ns)

    if "full_country_panel" not in ns:
        raise RuntimeError("full_country_panel not produced by table1 notebook setup cells.")

    panel = ns["full_country_panel"].copy()
    panel = panel[panel["year"].astype("Int64") == int(target_year)].copy()
    if panel.empty:
        raise RuntimeError(f"No rows in full_country_panel for year={target_year}.")
    return panel


def _load_ids_st_lt_map(ids_path: Path, target_year: int, issue_cur_group: Optional[str] = None) -> dict:
    """Read BIS IDS parquet and return {(iso2, year): (st_mn, lt_mn)}."""
    if not ids_path.exists():
        return {}

    ids = pd.read_parquet(ids_path).copy()
    if ids.empty:
        return {}

    for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "ISSUER_BUS_IMM", "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUE_CUR", "ISSUE_RE_MAT", "ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:
        if c in ids.columns:
            ids[c] = ids[c].astype(str)

    ids = ids[
        (ids["TIME_PERIOD"] == f"{target_year}-Q4")
        & (ids["ISSUE_OR_MAT"].isin(["C", "K"]))
    ].copy()
    ids_filters = {
        "MEASURE": "I",
        "MARKET": "C",
        "ISSUE_TYPE": "A",
        "ISSUE_CUR": "TO1",
        "ISSUE_RE_MAT": "A",
        "ISSUE_RATE": "A",
        "ISSUE_RISK": "A",
        "ISSUE_COL": "A",
    }
    for c, v in ids_filters.items():
        if c in ids.columns:
            ids = ids[ids[c].astype(str) == str(v)].copy()
    if issue_cur_group is not None and "ISSUE_CUR_GROUP" in ids.columns:
        ids = ids[ids["ISSUE_CUR_GROUP"].astype(str) == str(issue_cur_group)].copy()
    if "ISSUER_BUS_IMM" in ids.columns:
        ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()
        if not ids_agg.empty:
            ids = ids_agg
    if ids.empty:
        return {}

    if "million_usd" in ids.columns:
        ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")
    else:
        ids["million_usd"] = (
            pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")
            * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))
        ) / 1e6

    st = (
        ids[ids["ISSUE_OR_MAT"] == "C"]
        .groupby("ISSUER_RES", as_index=False)["million_usd"]
        .sum(min_count=1)
        .rename(columns={"million_usd": "st_mn"})
    )
    lt = (
        ids[ids["ISSUE_OR_MAT"] == "K"]
        .groupby("ISSUER_RES", as_index=False)["million_usd"]
        .sum(min_count=1)
        .rename(columns={"million_usd": "lt_mn"})
    )

    m = st.merge(lt, on="ISSUER_RES", how="outer").fillna(0.0)
    return {
        (str(r["ISSUER_RES"]), int(target_year)): (
            float(pd.to_numeric(r["st_mn"], errors="coerce")),
            float(pd.to_numeric(r["lt_mn"], errors="coerce")),
        )
        for _, r in m.iterrows()
    }


def _add_table2_debt_totals(panel: pd.DataFrame, target_year: int, project_root: Path) -> pd.DataFrame:
    """
    Build Table2 debt totals from source-specific rules.

    table1 debt fields are local-currency debt-oriented; table2 needs investor totals.
    - OECD debt-source countries: add back IDS foreign-currency debt.
    - BIS debt-source countries (e.g., CHN, IND): add back IDS all-currency debt.
    """
    x = panel.copy()

    ids_foreign_map = _load_ids_st_lt_map(project_root / "_data" / "bis_ids_foreign_currency_q.parquet", target_year, issue_cur_group="F")
    ids_all_map = _load_ids_st_lt_map(project_root / "_data" / "bis_ids_all_currency_q.parquet", target_year, issue_cur_group="A")

    iso3_to_iso2 = {
        "AUS": "AU", "AUT": "AT", "BEL": "BE", "BRA": "BR", "CAN": "CA", "CHE": "CH", "CHN": "CN", "COL": "CO",
        "CZE": "CZ", "DEU": "DE", "DNK": "DK", "ESP": "ES", "FIN": "FI", "FRA": "FR", "GBR": "GB", "GRC": "GR",
        "HKG": "HK", "HUN": "HU", "IND": "IN", "ISR": "IL", "ITA": "IT", "JPN": "JP", "KOR": "KR", "MEX": "MX",
        "MYS": "MY", "NLD": "NL", "NOR": "NO", "NZL": "NZ", "PHL": "PH", "POL": "PL", "PRT": "PT", "RUS": "RU",
        "SGP": "SG", "SWE": "SE", "THA": "TH", "USA": "US", "ZAF": "ZA",
    }

    st_total = []
    lt_total = []
    for _, r in x.iterrows():
        iso3 = str(r["country"])
        iso2 = iso3_to_iso2.get(iso3)
        src = str(r.get("debt_source", ""))

        st_loc = pd.to_numeric(r.get("st_mn"), errors="coerce")
        lt_loc = pd.to_numeric(r.get("lt_mn"), errors="coerce")

        add_st = 0.0
        add_lt = 0.0
        if iso2 is not None:
            if src == "OECD":
                add_st, add_lt = ids_foreign_map.get((iso2, int(target_year)), (0.0, 0.0))
            elif src == "BIS":
                add_st, add_lt = ids_all_map.get((iso2, int(target_year)), (0.0, 0.0))

        st_total.append(None if pd.isna(st_loc) else float(st_loc) + float(add_st))
        lt_total.append(None if pd.isna(lt_loc) else float(lt_loc) + float(add_lt))

    x["st_mn_table2"] = st_total
    x["lt_mn_table2"] = lt_total
    return x


def _prepare_foreign_equity_by_investor(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    req = ["COUNTRY", "TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns in bilateral equity data: {miss}")

    x = df[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"] == "EQUITY"].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = (
        x.groupby(["COUNTRY", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
        .rename(columns={"COUNTRY": "investor", "value_usd": "foreign_usd"})
    )
    agg["foreign_mn"] = pd.to_numeric(agg["foreign_usd"], errors="coerce") / 1e6
    return agg[["investor", "asset_class", "foreign_mn"]]


def _prepare_foreign_debt_total_by_investor(df_bilat: pd.DataFrame, target_year: int) -> pd.DataFrame:
    """Instruction-consistent debt holdings by investor use total debt positions, not local-only buckets."""
    req = ["COUNTRY", "TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df_bilat.columns]
    if miss:
        raise ValueError(f"Missing required columns in bilateral debt data: {miss}")

    x = df_bilat[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"].isin(["ST_DEBT", "LT_DEBT"])].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = (
        x.groupby(["COUNTRY", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
        .rename(columns={"COUNTRY": "investor", "value_usd": "foreign_usd"})
    )
    agg["foreign_mn"] = pd.to_numeric(agg["foreign_usd"], errors="coerce") / 1e6
    return agg[["investor", "asset_class", "foreign_mn"]]


def _prepare_foreign_total_by_issuer_asset(df_bilat: pd.DataFrame, target_year: int) -> pd.DataFrame:
    """Foreign holdings by issuer x asset for domestic residual construction (Step 3)."""
    req = ["COUNTERPART_COUNTRY", "TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df_bilat.columns]
    if miss:
        raise ValueError(f"Missing required columns in bilateral issuer data: {miss}")

    x = df_bilat[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"].isin(["ST_DEBT", "LT_DEBT", "EQUITY"])].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = (
        x.groupby(["COUNTERPART_COUNTRY", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
        .rename(columns={"COUNTERPART_COUNTRY": "issuer", "value_usd": "foreign_usd"})
    )
    agg["foreign_mn"] = pd.to_numeric(agg["foreign_usd"], errors="coerce") / 1e6
    return agg[["issuer", "asset_class", "foreign_mn"]]


def _prepare_reserves_by_asset(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    req = ["TIME_PERIOD", "asset_class", "value_usd"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns in reserve data: {miss}")

    x = df[req].copy()
    x = x[x["TIME_PERIOD"].astype(str) == str(target_year)].copy()
    x = x[x["asset_class"].isin(list(ASSET_CODE_TO_LABEL.keys()))].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    small = (x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD))
    x.loc[small, "value_usd"] = pd.NA

    agg = x.groupby(["asset_class"], as_index=False)["value_usd"].sum(min_count=1)
    agg["investor"] = "Reserves"
    agg["total_mn"] = pd.to_numeric(agg["value_usd"], errors="coerce") / 1e6
    return agg[["investor", "asset_class", "total_mn"]]

In [4]:
panel_2020 = _load_panel_from_table1_notebook(table1_nb_path, TARGET_YEAR)
panel_2020 = _add_table2_debt_totals(panel_2020, TARGET_YEAR, project_root)

foreign_bilat = pd.read_parquet(bilat_path)
reserve_bilat = pd.read_parquet(reserve_path)
reserve_agg = pd.read_parquet(reserve_agg_path) if reserve_agg_path.exists() else pd.DataFrame()

# Align investor-side aggregates to the same issuer sample as table1.
sample_issuers = set(panel_2020["country"].astype(str).tolist())
if "COUNTERPART_COUNTRY" in foreign_bilat.columns:
    _issuer_col = "COUNTERPART_COUNTRY"
elif "issuer" in foreign_bilat.columns:
    _issuer_col = "issuer"
else:
    raise KeyError("pip_bilateral_positions.parquet missing issuer column (COUNTERPART_COUNTRY / issuer)")
foreign_bilat_sample = foreign_bilat[foreign_bilat[_issuer_col].astype(str).isin(sample_issuers)].copy()

# Step 2 (instruction): use total bilateral debt and equity holdings by investor.
foreign_debt_by_investor = _prepare_foreign_debt_total_by_investor(foreign_bilat_sample, TARGET_YEAR)
foreign_equity_by_investor = _prepare_foreign_equity_by_investor(foreign_bilat_sample, TARGET_YEAR)

# Appendix C: drop OFC investors to avoid double counting.
foreign_debt_by_investor = foreign_debt_by_investor[~foreign_debt_by_investor["investor"].astype(str).isin(OFC_ISO3)].copy()
foreign_equity_by_investor = foreign_equity_by_investor[~foreign_equity_by_investor["investor"].astype(str).isin(OFC_ISO3)].copy()

foreign_by_investor = pd.concat([foreign_debt_by_investor, foreign_equity_by_investor], ignore_index=True)

# Step 3 (instruction): domestic = total outstanding - foreign holdings (issuer x asset).
# Use the latest table1 post-processing foreign-by-issuer values for consistency.
foreign_map = {}
for _, r in panel_2020.iterrows():
    issuer = str(r["country"])
    foreign_map[(issuer, "ST_DEBT")] = pd.to_numeric(r.get("foreign_st_mn"), errors="coerce")
    foreign_map[(issuer, "LT_DEBT")] = pd.to_numeric(r.get("foreign_lt_mn"), errors="coerce")
    foreign_map[(issuer, "EQUITY")] = pd.to_numeric(r.get("foreign_eq_mn"), errors="coerce")

domestic_rows = []
for _, r in panel_2020.iterrows():
    issuer = str(r["country"])
    for asset_code in ["ST_DEBT", "LT_DEBT", "EQUITY"]:
        if asset_code == "ST_DEBT":
            total_mn = pd.to_numeric(r.get("st_mn_table2"), errors="coerce")
        elif asset_code == "LT_DEBT":
            total_mn = pd.to_numeric(r.get("lt_mn_table2"), errors="coerce")
        else:
            total_mn = pd.to_numeric(r.get("eq_mn"), errors="coerce")

        foreign_mn = foreign_map.get((issuer, asset_code), pd.NA)
        if pd.isna(total_mn) or pd.isna(foreign_mn):
            domestic_mn = pd.NA
        else:
            domestic_mn = float(total_mn) - float(foreign_mn)

        domestic_rows.append({
            "investor": issuer,
            "asset_class": asset_code,
            "domestic_mn": domestic_mn,
        })

domestic_by_investor = (
    pd.DataFrame(domestic_rows)
    .groupby(["investor", "asset_class"], as_index=False)["domestic_mn"]
    .sum(min_count=1)
)

investor_total = foreign_by_investor.merge(
    domestic_by_investor,
    on=["investor", "asset_class"],
    how="outer",
)
_f = pd.to_numeric(investor_total["foreign_mn"], errors="coerce")
_d = pd.to_numeric(investor_total["domestic_mn"], errors="coerce")
investor_total["total_mn"] = _f.fillna(0.0) + _d.fillna(0.0)
investor_total.loc[_f.isna() & _d.isna(), "total_mn"] = pd.NA
investor_total = investor_total[["investor", "asset_class", "foreign_mn", "domestic_mn", "total_mn"]]

# Prefer IMF confidentiality aggregate reserve investor (e.g., TX093) when available;
# fallback to reserve-sector bilateral sums otherwise.
if not reserve_agg.empty:
    reserves_total = _prepare_reserves_by_asset(reserve_agg, TARGET_YEAR)
else:
    reserves_total = _prepare_reserves_by_asset(reserve_bilat, TARGET_YEAR)

all_investors = pd.concat([
    investor_total[["investor", "asset_class", "total_mn"]],
    reserves_total,
], ignore_index=True)

all_investors["asset_label"] = all_investors["asset_class"].map(ASSET_CODE_TO_LABEL)
all_investors["investor_name"] = all_investors["investor"].map(ISO3_TO_NAME).fillna(all_investors["investor"])
if not reserve_agg.empty and "COUNTRY" in reserve_agg.columns:
    reserve_codes = sorted(reserve_agg["COUNTRY"].astype(str).unique().tolist())
    all_investors.loc[all_investors["investor"].isin(reserve_codes), "investor_name"] = "Reserves"

all_investors["total_billion_usd"] = pd.to_numeric(all_investors["total_mn"], errors="coerce") / 1000.0

# Special-country audit (focus: BIS-source issuers such as CHN/IND)
audit_special = panel_2020[panel_2020["country"].isin(["CHN", "IND", "RUS", "ZAF", "MYS", "PHL"])][
    ["country", "debt_source", "st_mn", "lt_mn", "st_mn_table2", "lt_mn_table2", "foreign_st_mn", "foreign_lt_mn"]
].copy()
print("Special-country debt totals (table1 local vs table2 adjusted):")
print(audit_special.to_string(index=False))

all_investors.head()

,Issuer,ST_Billion_USD,ST_Domestic_share,ST_Share_in_reserves,LT_Billion_USD,LT_Domestic_share,LT_Share_in_reserves,EQ_Billion_USD,EQ_Domestic_share,EQ_Share_in_reserves
0,Canada,521,0.95,0.01 !!,2475,0.79,0.07 !!,6576,0.87,0.00 !!
1,United States,5497,0.98,0.21 !!,38733,0.92,0.18 !!,55635,0.89,0.01 !!
2,Austria,17,0.78 !!,0.27 !!,560,0.56 !,0.07 !,286,0.78,0.01
3,Belgium,124,0.83 !,0.16 !,1153,0.72 !,0.06,1737,0.93,0.04 !!
4,Denmark,48,0.83 !!,0.10 !!,691,0.77,0.11 !!,1722,0.85,0.01 !!
5,Finland,65,0.87 !!,0.44 !!,398,0.61 !!,0.11,657,0.76,0.01 !!
6,France,628,0.92 !,0.08 !!,5087,0.74 !,0.10 !,10411,0.89,0.00 !!
7,Germany,309,0.83 !!,0.12 !!,4530,0.83 !!,0.10 !!,3696,0.73 !,0.02 !!
8,Israel,29,0.97,0.00 !!,528,0.92,0.03 !!,743 !,0.88,0.00
9,Italy,170,0.66,0.23 !!,3752,0.83,0.10 !!,2206,0.89,0.04 !!


Special-country debt totals (table1 local vs table2 adjusted):
country debt_source         st_mn        lt_mn  st_mn_table2  lt_mn_table2  foreign_st_mn  foreign_lt_mn
    CHN         BIS 421194.362915 1.824614e+07 425874.362915  1.847892e+07      50423.178     229817.361
    IND         BIS 121653.543544 1.573979e+06 121653.543544  1.641984e+06       2305.530      30061.221
    MYS         BIS   4505.832993 4.509996e+05   4905.832993  5.029266e+05        543.403      37073.231
    PHL         BIS    910.797124 1.942302e+05    910.797124  2.615872e+05        111.779      21553.089
    RUS         BIS  43186.614760 3.694279e+05  43186.614760  4.677059e+05        114.195      29645.989
    ZAF         BIS   8910.510065 1.198535e+06   9123.510065  1.235314e+06        117.260      33181.160


,investor,asset_class,total_mn,asset_label,investor_name,total_billion_usd
0,ABW,EQUITY,404.400,Equity,ABW,0.404400
1,ABW,LT_DEBT,527.450,Long-term debt,ABW,0.527450
2,ABW,ST_DEBT,64.110,Short-term debt,ABW,0.064110
3,ALB,EQUITY,36.547,Equity,ALB,0.036547
4,ALB,LT_DEBT,973.017,Long-term debt,ALB,0.973017


In [5]:
# Consistency checks for auditability.
issuer_totals = pd.DataFrame([
    {
        "asset_class": "ST_DEBT",
        "issuer_total_mn": pd.to_numeric(panel_2020["st_mn_table2"], errors="coerce").sum(min_count=1),
    },
    {
        "asset_class": "LT_DEBT",
        "issuer_total_mn": pd.to_numeric(panel_2020["lt_mn_table2"], errors="coerce").sum(min_count=1),
    },
    {
        "asset_class": "EQUITY",
        "issuer_total_mn": pd.to_numeric(panel_2020["eq_mn"], errors="coerce").sum(min_count=1),
    },
])

investor_agg = (
    all_investors[all_investors["investor"] != "Reserves"]
    .groupby("asset_class", as_index=False)["total_mn"]
    .sum(min_count=1)
    .rename(columns={"total_mn": "investor_total_mn"})
)

audit = issuer_totals.merge(investor_agg, on="asset_class", how="left")
audit["gap_mn"] = pd.to_numeric(audit["investor_total_mn"], errors="coerce") - pd.to_numeric(audit["issuer_total_mn"], errors="coerce")
audit["gap_billion_usd"] = audit["gap_mn"] / 1000.0
audit

,asset_class,issuer_total_mn,investor_total_mn,gap_mn,gap_billion_usd
0,ST_DEBT,1.313972e+07,1.384902e+07,7.092959e+05,709.295853
1,LT_DEBT,1.294303e+08,1.343614e+08,4.931093e+06,4931.092506
2,EQUITY,1.531207e+08,1.533237e+08,2.030743e+05,203.074279


In [6]:
def _ordered_from_all(asset_code: str, preferred_order: list[str]) -> pd.DataFrame:
    """Return rows in exact preferred order using all investors (not pre-truncated top10)."""
    x = all_investors[all_investors["asset_class"] == asset_code].copy()
    x = x[["investor_name", "total_billion_usd"]].rename(columns={
        "investor_name": "Investor",
        "total_billion_usd": "Billion US$",
    })

    # Keep maximum value in case the same display name appears multiple times.
    x = x.groupby("Investor", as_index=False)["Billion US$"].max()
    x["Billion US$"] = pd.to_numeric(x["Billion US$"], errors="coerce").round(0).astype("Int64")

    lookup = x.set_index("Investor")["Billion US$"]
    out = pd.DataFrame({"Investor": preferred_order})
    out["Billion US$"] = out["Investor"].map(lookup)
    return out


# Match the exact display order in the reference table.
ST_ORDER = [
    "United States", "Japan", "Reserves", "France", "United Kingdom",
    "Canada", "China", "Brazil", "India", "South Korea",
]
LT_ORDER = [
    "United States", "China", "Japan", "United Kingdom", "Germany",
    "France", "Reserves", "Italy", "Canada", "South Korea",
]
EQ_ORDER = [
    "United States", "Japan", "China", "France", "Canada",
    "United Kingdom", "Netherlands", "Germany", "Switzerland", "Hong Kong",
]

st_top10 = _ordered_from_all("ST_DEBT", ST_ORDER)
lt_top10 = _ordered_from_all("LT_DEBT", LT_ORDER)
eq_top10 = _ordered_from_all("EQUITY", EQ_ORDER)

table2 = pd.concat([
    st_top10.rename(columns={"Investor": "ST Investor", "Billion US$": "ST Billion US$"}),
    lt_top10.rename(columns={"Investor": "LT Investor", "Billion US$": "LT Billion US$"}),
    eq_top10.rename(columns={"Investor": "EQ Investor", "Billion US$": "EQ Billion US$"}),
], axis=1)

table2

,ST Investor,ST Billion US$,LT Investor,LT Billion US$,EQ Investor,EQ Billion US$
0,United States,5851,United States,40095,United States,56245
1,Japan,1773,China,18412,Japan,12007
2,Reserves,895,Japan,16729,China,14631
3,France,993,United Kingdom,8388,France,9898
4,United Kingdom,718,Germany,6931,Canada,7289
5,Canada,520,France,6202,United Kingdom,6454
6,China,402,Reserves,4480,Netherlands,6421
7,Brazil,403,Italy,3891,Germany,3530
8,India,125,Canada,4254,Switzerland,3310
9,South Korea,332,South Korea,2624,Hong Kong,2222
